# Credit Card Default Prediction

**Goal:** Predict whether a client will default on their next credit card payment.

**Dataset:** UCI Default of Credit Card Clients (30,000 records, 24 features)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, roc_curve
)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)

## 1. Load and Clean Data

In [ ]:
df = pd.read_csv("../data/credit_default.csv")
df.columns = [
    "id", "credit_limit", "sex", "education", "marriage", "age",
    "pay_sep", "pay_aug", "pay_jul", "pay_jun", "pay_may", "pay_apr",
    "bill_sep", "bill_aug", "bill_jul", "bill_jun", "bill_may", "bill_apr",
    "paid_sep", "paid_aug", "paid_jul", "paid_jun", "paid_may", "paid_apr",
    "default"
]
df = df.drop(columns=["id"])
print(f"Shape: {df.shape}")
df.head()

In [ ]:
df.info()
print("\nMissing values:", df.isnull().sum().sum())
print(f"\nDefault rate: {df['default'].mean():.1%}")

## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

df["default"].value_counts().plot.bar(ax=axes[0], color=["steelblue", "coral"])
axes[0].set_title("Default Distribution")
axes[0].set_xticklabels(["No Default", "Default"], rotation=0)

df["credit_limit"].hist(bins=50, ax=axes[1], color="steelblue", edgecolor="white")
axes[1].set_title("Credit Limit Distribution")

df["age"].hist(bins=30, ax=axes[2], color="steelblue", edgecolor="white")
axes[2].set_title("Age Distribution")

plt.tight_layout()
plt.savefig("../figures/eda_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df.groupby("education")["default"].mean().plot.bar(ax=axes[0], color="coral")
axes[0].set_title("Default Rate by Education")
axes[0].set_ylabel("Default Rate")

df.groupby("marriage")["default"].mean().plot.bar(ax=axes[1], color="coral")
axes[1].set_title("Default Rate by Marriage Status")
axes[1].set_ylabel("Default Rate")

plt.tight_layout()
plt.savefig("../figures/eda_default_rates.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
pay_cols = ["pay_sep", "pay_aug", "pay_jul", "pay_jun", "pay_may", "pay_apr"]
corr = df[pay_cols + ["default"]].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap="RdBu_r", center=0, fmt=".2f")
plt.title("Correlation: Repayment Status vs Default")
plt.tight_layout()
plt.savefig("../figures/eda_correlation.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Feature Engineering and Split

In [ ]:
pay_cols = ["pay_sep", "pay_aug", "pay_jul", "pay_jun", "pay_may", "pay_apr"]
bill_cols = ["bill_sep", "bill_aug", "bill_jul", "bill_jun", "bill_may", "bill_apr"]
paid_cols = ["paid_sep", "paid_aug", "paid_jul", "paid_jun", "paid_may", "paid_apr"]

df["avg_bill"] = df[bill_cols].mean(axis=1)
df["avg_paid"] = df[paid_cols].mean(axis=1)
df["payment_ratio"] = df["avg_paid"] / (df["avg_bill"] + 1)
df["late_count"] = (df[pay_cols] > 0).sum(axis=1)
df["utilization"] = df["avg_bill"] / (df["credit_limit"] + 1)
df["max_late"] = df[pay_cols].max(axis=1)
df["pay_trend"] = df["pay_sep"] - df["pay_apr"]
df["bill_std"] = df[bill_cols].std(axis=1)
df["paid_std"] = df[paid_cols].std(axis=1)
df["total_paid"] = df[paid_cols].sum(axis=1)
df["total_bill"] = df[bill_cols].sum(axis=1)
df["total_payment_ratio"] = df["total_paid"] / (df["total_bill"] + 1)
df["ever_2months_late"] = (df[pay_cols] >= 2).any(axis=1).astype(int)
df["credit_per_age"] = df["credit_limit"] / (df["age"] + 1)
df["avg_pay_status"] = df[pay_cols].mean(axis=1)
df["pay_sep_sq"] = df["pay_sep"] ** 2
df["limit_util_interaction"] = df["credit_limit"] * df["utilization"]
df["recent_late"] = (df[["pay_sep", "pay_aug", "pay_jul"]] > 0).sum(axis=1)
df["old_late"] = (df[["pay_jun", "pay_may", "pay_apr"]] > 0).sum(axis=1)
df["late_trend"] = df["recent_late"] - df["old_late"]
df["bill_change"] = df["bill_sep"] - df["bill_apr"]
df["paid_to_limit"] = df["total_paid"] / (df["credit_limit"] + 1)
df["max_bill"] = df[bill_cols].max(axis=1)
df["min_paid"] = df[paid_cols].min(axis=1)
df["ever_3months_late"] = (df[pay_cols] >= 3).any(axis=1).astype(int)

df = df.fillna(0)

print(f"Features: {df.shape[1] - 1}")
df.head()

In [ ]:
X = df.drop(columns=["default"])
y = df["default"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")
print(f"Default rate — Train: {y_train.mean():.1%} | Test: {y_test.mean():.1%}")

## 4. Model Training

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, max_depth=12, min_samples_leaf=5, random_state=42
    ),
    "XGBoost": XGBClassifier(
        n_estimators=500, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric="logloss", random_state=42
    ),
}

results = {}
for name, model in models.items():
    if name == "Logistic Regression":
        model.fit(X_train_scaled, y_train)
        y_prob = model.predict_proba(X_test_scaled)[:, 1]
        y_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_prob = model.predict_proba(X_test)[:, 1]
        y_pred = model.predict(X_test)

    auc = roc_auc_score(y_test, y_prob)
    acc = (y_pred == y_test).mean()
    results[name] = {"model": model, "y_prob": y_prob, "y_pred": y_pred, "auc": auc}
    print(f"{name}: ROC-AUC = {auc:.4f} | Accuracy = {acc:.1%}")
    print(classification_report(y_test, y_pred, target_names=["No Default", "Default"]))
    print("-" * 60)

## 5. Model Comparison

In [ ]:
plt.figure(figsize=(8, 6))
for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res["y_prob"])
    plt.plot(fpr, tpr, label=f"{name} (AUC={res['auc']:.3f})")

plt.plot([0, 1], [0, 1], "k--", alpha=0.3)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves — Model Comparison")
plt.legend()
plt.tight_layout()
plt.savefig("../figures/roc_curves.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
best_name = max(results, key=lambda k: results[k]["auc"])
best = results[best_name]

cm = confusion_matrix(y_test, best["y_pred"])
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["No Default", "Default"],
            yticklabels=["No Default", "Default"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title(f"Confusion Matrix — {best_name}")
plt.tight_layout()
plt.savefig("../figures/confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Feature Importance

In [ ]:
xgb_model = results["XGBoost"]["model"]
importance = pd.Series(xgb_model.feature_importances_, index=X.columns)
importance = importance.sort_values(ascending=True)

plt.figure(figsize=(8, 8))
importance.tail(15).plot.barh(color="steelblue")
plt.title("Top 15 Features — XGBoost Importance")
plt.xlabel("Importance")
plt.tight_layout()
plt.savefig("../figures/feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Key Findings

1. **Repayment history** is the strongest predictor of default — clients who paid late in recent months are far more likely to default again.
2. **Payment ratio** (how much of the bill is actually paid) is a strong engineered feature.
3. **Credit limit** matters — higher limits correlate with lower default rates (banks already screen these clients).
4. **XGBoost** typically performs best, followed by Random Forest, then Logistic Regression.
5. The dataset is imbalanced (~22% default rate), which is realistic for credit risk.